In [0]:
# Databricks notebook source

# 01 - Bronze Ingestion

Ingestão incremental simulada dos dados de corridas de táxi da NYC TLC.
Cada execução deste notebook processa **um mês** específico, definido pelos widgets `year` e `month`.
A carga é idempotente: reprocessar o mesmo mês não gera duplicidade, graças ao `MERGE INTO` baseado
em um hash de linha. Uma tabela de controle (`_ingestion_log`) registra o histórico de execuções.
Fonte: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page


Configuração e widgets

In [0]:
dbutils.widgets.text("year", "2024", "Ano (YYYY)")
dbutils.widgets.text("month", "01", "Mês (MM)")
 
year = dbutils.widgets.get("year")
month = dbutils.widgets.get("month")
year_month = f"{year}-{month}"
 
CATALOG = "nyc_taxi"
BRONZE_SCHEMA = "bronze"
VOLUME_PATH = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/raw_files"
 
BRONZE_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.trips_bronze"
INGESTION_LOG_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}._ingestion_log"
 
SOURCE_URL = f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{year_month}.parquet"
LOCAL_FILE_PATH = f"{VOLUME_PATH}/yellow_tripdata_{year_month}.parquet"
 
print(f"Processando mês: {year_month}")
print(f"Origem: {SOURCE_URL}")
print(f"Destino no volume: {LOCAL_FILE_PATH}")

## Criação das tabelas de controle (idempotente)
Roda uma vez por ambiente novo; `CREATE TABLE IF NOT EXISTS` garante que reexecuções não quebram nada.

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {BRONZE_TABLE} (
    row_hash STRING,
    year_month STRING,
    VendorID INT,
    tpep_pickup_datetime TIMESTAMP,
    tpep_dropoff_datetime TIMESTAMP,
    passenger_count DOUBLE,
    trip_distance DOUBLE,
    RatecodeID DOUBLE,
    store_and_fwd_flag STRING,
    PULocationID INT,
    DOLocationID INT,
    payment_type BIGINT,
    fare_amount DOUBLE,
    extra DOUBLE,
    mta_tax DOUBLE,
    tip_amount DOUBLE,
    tolls_amount DOUBLE,
    improvement_surcharge DOUBLE,
    total_amount DOUBLE,
    congestion_surcharge DOUBLE,
    airport_fee DOUBLE,
    _source_file STRING,
    _ingested_at TIMESTAMP
)
USING DELTA
PARTITIONED BY (year_month)
""")
 
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {INGESTION_LOG_TABLE} (
    year_month STRING,
    status STRING,
    rows_processed BIGINT,
    source_file STRING,
    started_at TIMESTAMP,
    finished_at TIMESTAMP
)
USING DELTA
""")
 
print("Tabelas prontas.")

## Checagem de idempotência

Se o mês já foi carregado com sucesso, o notebook para aqui — evita reprocessamento desnecessário.
Para forçar reprocessamento, delete a linha correspondente em `_ingestion_log` antes de rodar.

In [0]:
from pyspark.sql import functions as F
 
already_ingested = (
    spark.table(INGESTION_LOG_TABLE)
    .filter((F.col("year_month") == year_month) & (F.col("status") == "SUCCESS"))
    .count()
    > 0
)
 
if already_ingested:
    dbutils.notebook.exit(f"Mês {year_month} já foi ingerido com sucesso. Nada a fazer.")

## Obtenção do arquivo de origem
 
Tenta baixar o parquet diretamente da NYC TLC. O Free Edition do Databricks restringe acesso de
saída a um conjunto limitado de domínios, sem lista pública — então o download pode falhar por
bloqueio de rede, não só por erro de conexão comum.
 
Se o download falhar, o notebook verifica se o arquivo já existe no volume (upload manual como
fallback) antes de desistir.

In [0]:
import os
import requests
from datetime import datetime
 
started_at = datetime.now()
 
def file_exists_in_volume(path: str) -> bool:
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False
    
download_ok = False

if not file_exists_in_volume(LOCAL_FILE_PATH):
    try:
        print(f"Arquivo não encontrado no volume. Tentando download de {SOURCE_URL} ...")
        response = requests.get(SOURCE_URL, timeout=60)
        response.raise_for_status()
 
        with open(LOCAL_FILE_PATH, "wb") as f:
            f.write(response.content)
 
        download_ok = True
        print("Download concluído com sucesso.")
 
    except Exception as e:
        print(f"Falha no download direto: {e}")
        print(
            "Provável bloqueio de rede do Free Edition (domínio não liberado). "
            f"Faça upload manual do arquivo 'yellow_tripdata_{year_month}.parquet' "
            f"para o volume em {VOLUME_PATH} e rode este notebook novamente."
        )
else:
    print("Arquivo já presente no volume (upload manual ou execução anterior). Pulando download.")
    download_ok = True

In [0]:
if not file_exists_in_volume(LOCAL_FILE_PATH):
    log_row = spark.createDataFrame(
        [(year_month, "FAILED", 0, LOCAL_FILE_PATH, started_at, datetime.now())],
        schema="year_month STRING, status STRING, rows_processed BIGINT, source_file STRING, started_at TIMESTAMP, finished_at TIMESTAMP",
    )
    log_row.write.mode("append").saveAsTable(INGESTION_LOG_TABLE)
    dbutils.notebook.exit(f"Ingestão de {year_month} interrompida: arquivo indisponível.")

## Leitura, hash de linha e MERGE INTO

O `row_hash` é calculado a partir das colunas de negócio, já que o dataset não tem uma chave
primária natural. Ele garante que reprocessar o mesmo arquivo não duplica linhas no MERGE.

In [0]:
raw_df = spark.read.parquet(LOCAL_FILE_PATH)
 
business_columns = [c for c in raw_df.columns]
 
df = (
    raw_df
    .withColumn("row_hash", F.sha2(F.concat_ws("||", *[F.col(c).cast("string") for c in business_columns]), 256))
    .withColumn("year_month", F.lit(year_month))
    .withColumn("_source_file", F.lit(LOCAL_FILE_PATH))
    .withColumn("_ingested_at", F.current_timestamp())
)
 
row_count = df.count()
print(f"Linhas lidas do arquivo: {row_count}")
 
df.createOrReplaceTempView("staged_trips")

In [0]:
merge_result = spark.sql(f"""
MERGE INTO {BRONZE_TABLE} AS target
USING staged_trips AS source
ON target.row_hash = source.row_hash AND target.year_month = source.year_month
WHEN NOT MATCHED THEN INSERT *
""")

display(merge_result)

## Atualização do log de ingestão

In [0]:
finished_at = datetime.now()
 
log_row = spark.createDataFrame(
    [(year_month, "SUCCESS", row_count, LOCAL_FILE_PATH, started_at, finished_at)],
    schema="year_month STRING, status STRING, rows_processed BIGINT, source_file STRING, started_at TIMESTAMP, finished_at TIMESTAMP",
)
log_row.write.mode("append").saveAsTable(INGESTION_LOG_TABLE)
 
print(f"Ingestão de {year_month} concluída: {row_count} linhas processadas.")

## Verificação rápida

In [0]:
display(spark.table(INGESTION_LOG_TABLE).orderBy("year_month"))